## Data Export

In [1]:
import findspark
findspark.init() 

from pyspark.sql import SparkSession

# Connect Spark to HDFS cluster
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Vietnam_THOR_HDFS") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()



26/03/10 21:04:45 WARN Utils: Your hostname, Anmols-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.83 instead (on interface en0)
26/03/10 21:04:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 21:04:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Load the file from the HDFS path

path = "hdfs://localhost:9000/user/anmolkhatri/thor_data/THOR_Vietnam_7_31_2013.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .option("ignoreLeadingWhiteSpace", "true") \
    .option("ignoreTrailingWhiteSpace", "true") \
    .load(path)

print("Success! Spark is now running.")

df.show()

Success! Spark is now running.


26/03/10 13:29:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+--------+--------+-----------+-------+--------------------+----------+----------------+-------------+------------+------+--------+-----------------+-------------+---------+-----+-------------------+----------------+------------------+---------------+-------------+-----------------+----------------+-------------+-----------------+-----------------+--------+---------+-----------+---------------+---------------+-------------------+------------------+---------------+-------------------+-------------------+---------------+-------------------+----------------------+-------------+------------------+-----+-----------+----------+----------+-------------+------------+--------------------+----------------+--------------------+------------------+-------------------+-------------------+-----------------------+---------------------+----------------------+--------------+-----------------+---------------+---------------+-----------+------------+-------------+------------+--------------+-

In [3]:
df.printSchema()

root
 |-- SourceRecord: string (nullable = true)
 |-- SourceID: integer (nullable = true)
 |-- MSNDate: integer (nullable = true)
 |-- Conflict: string (nullable = true)
 |-- Theater: string (nullable = true)
 |-- CountryFlyingMission: string (nullable = true)
 |-- MilService: string (nullable = true)
 |-- NumberedAirForce: string (nullable = true)
 |-- AirForceGroup: string (nullable = true)
 |-- AirForceSQDN: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Callsign: string (nullable = true)
 |-- Aircraft_Original: string (nullable = true)
 |-- Aircraft_Root: string (nullable = true)
 |-- MissionID: string (nullable = true)
 |-- MFUNC: string (nullable = true)
 |-- MFUNC_DESC: string (nullable = true)
 |-- ServiceSupported: double (nullable = true)
 |-- OperationSupported: string (nullable = true)
 |-- TakeoffLocation: string (nullable = true)
 |-- TakeoffLatDeg: integer (nullable = true)
 |-- TakeoffLatDecimal: double (nullable = true)
 |-- TakeoffLatDD_DDD: double 

## Data Cleaning

In [4]:
from pyspark.sql import functions as F

# 1. Drop rows missing critical spatial data for Tableau map
# This ensures the 'Dynamic Time-Series Map' has no errors 
df_cleaned = df.filter(
    df.TgtLatDD_DDD_WGS84.isNotNull() & 
    df.TgtLonDDD_DDD_WGS84.isNotNull()
)

In [5]:
# 3. Fill missing Aircraft counts with the median (or 1 )
df_cleaned = df_cleaned.na.fill({"NumOfAcft": 1})

print(f"Original Rows: {df.count()} | Cleaned Rows: {df_cleaned.count()}")

[Stage 6:>                                                          (0 + 1) / 1]

Original Rows: 4670422 | Cleaned Rows: 4670416


## Feature Engineering
### Convert Mission function and Aircraft Type strings to Indices

In [6]:
from pyspark.ml.feature import StringIndexer

# 1. Temporal Features: Extract Year and Month from MSNDate (YYYYMMDD format)
df_cleaned = df_cleaned.withColumn("Year", F.substring(F.col("MSNDate").cast("string"), 1, 4)) \
                       .withColumn("Month", F.substring(F.col("MSNDate").cast("string"), 5, 2))


In [7]:
# 2. Categorical Indexing: Transform strings 
# Includes Aircraft_Root, MFUNC_DESC, and MilService 
for col_name in ["Aircraft_Root", "MFUNC_DESC", "MilService", "TgtCountry"]:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_Index", handleInvalid="keep")
    df_cleaned = indexer.fit(df_cleaned).transform(df_cleaned)

In [8]:
print(f"Columns in Cleaned DF: {len(df_cleaned.columns)}") 

Columns in Cleaned DF: 91


## Tableu Exports for EDA

In [9]:
def save_for_tableau(dataframe, filename):
    dataframe.coalesce(1).write.mode("overwrite").option("header", "true") \
             .csv(f"file:///Users/anmolkhatri/Downloads/{filename}")

# --- 1. Dynamic War Progression (Time-Series Map) ---
save_for_tableau(
    df_cleaned.select("Year", "TgtCountry", "TgtLatDD_DDD_WGS84", "TgtLonDDD_DDD_WGS84"),
    "Tableau_War_Progression_Map"
)

# --- 2. Aircraft Performance (B-52 vs F-4 Comparison) ---
save_for_tableau(
    df_cleaned.filter(F.col("Aircraft_Root").isin(["B52", "F4"])) \
              .groupBy("Aircraft_Root", "Year") \
              .agg(F.sum("WeaponTypeWeight").alias("TotalWeight")),
    "Tableau_Aircraft_Performance"
)

# --- 3. Operational Share (Service Comparison: USAF, USN, USMC) ---
save_for_tableau(
    df_cleaned.groupBy("Year", "MilService").count(),
    "Tableau_Service_Operational_Share"
)

# --- 4. Mission Function (Donut Chart Data) ---
save_for_tableau(
    df_cleaned.groupBy("MFUNC_DESC").count(),
    "Tableau_Mission_Functions"
)

# --- 5. Regression EDA (Weight vs Planes Scatter Plot) ---
save_for_tableau(
    df_cleaned.select("NumOfAcft", "WeaponTypeWeight", "TgtCountry").limit(100000),
    "Tableau_Regression_EDA"
)

# --- 6. Temporal Gap Identification (Monthly Intensity) ---
# Specifically addressing the Nov 1967 gap in your proposal
save_for_tableau(
    df_cleaned.groupBy("Year", "Month").count(),
    "Tableau_Temporal_Intensity"
)

# --- 7. Regional Density (Country-wise Treemap) ---
save_for_tableau(
    df_cleaned.groupBy("TgtCountry", "MFUNC_DESC").count(),
    "Tableau_Regional_Density"
)

In [10]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Fix types and derive 'Year' from MSNDate (YYYYMMDD)
# We cast everything to Double for the ML models
df_fixed = df.withColumn("Year", (F.col("MSNDate") / 10000).cast("int")) \
             .withColumn("Month", F.substring(F.col("MSNDate").cast("string"), 5, 2))\
             .withColumn("lat", F.col("TgtLatDD_DDD_WGS84").cast("double")) \
             .withColumn("lon", F.col("TgtLonDDD_DDD_WGS84").cast("double")) \
             .withColumn("weight", F.col("WeaponTypeWeight").cast("double")) \
             .withColumn("planes", F.col("NumOfAcft").cast("double")) \
             .filter("lat IS NOT NULL AND lon IS NOT NULL AND weight > 0")

# 2. Re-Index categorical columns (handling potential existing column errors)
for c in ["Aircraft_Root", "MFUNC_DESC"]:
    out_col = f"{c}_Index"
    if out_col not in df_fixed.columns:
        indexer = StringIndexer(inputCol=c, outputCol=out_col, handleInvalid="keep")
        df_fixed = indexer.fit(df_fixed).transform(df_fixed)

# 3. Create the required feature vectors
# Clustering needs geo_features, Regression needs features_reg
geo_asm = VectorAssembler(inputCols=["lat", "lon"], outputCol="geo_features")
reg_asm = VectorAssembler(inputCols=["planes", "MFUNC_DESC_Index", "Aircraft_Root_Index"], 
                          outputCol="features_reg")

df_final_input = reg_asm.transform(geo_asm.transform(df_fixed))

## Data Preprocessing
### Here VectorAssembler is used to prepare data. Numerical indices are converted to vectors.

In [11]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Cast to Double and Filter
# We create a fresh 'df_ml_input' to avoid conflicts with previous versions
df_ml_input = df_cleaned.withColumn("lat", F.col("TgtLatDD_DDD_WGS84").cast("double")) \
                        .withColumn("lon", F.col("TgtLonDDD_DDD_WGS84").cast("double")) \
                        .withColumn("weight", F.col("WeaponTypeWeight").cast("double")) \
                        .withColumn("planes", F.col("NumOfAcft").cast("double")) \
                        .filter("lat IS NOT NULL AND lon IS NOT NULL AND weight > 0")

# 2. Safe Indexing: Check if the index columns already exist
# If they exist, we skip indexing. If not, we create them.
if "Aircraft_Root_Index" not in df_ml_input.columns:
    for col_name in ["Aircraft_Root", "MFUNC_DESC"]:
        indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_Index", handleInvalid="keep")
        df_ml_input = indexer.fit(df_ml_input).transform(df_ml_input)

# Verify the schema is ready
df_ml_input.select("lat", "lon", "weight", "planes", "Aircraft_Root_Index").show(5)

+---------+----------+------+------+-------------------+
|      lat|       lon|weight|planes|Aircraft_Root_Index|
+---------+----------+------+------+-------------------+
|  16.9025|106.014166| 750.0|   2.0|                3.0|
|19.602222|103.597222| 500.0|   2.0|                0.0|
|17.563611|105.756666|1388.0|   1.0|                7.0|
|12.280833|105.128888| 126.0|   2.0|                4.0|
|11.131666|105.498333| 250.0|   2.0|                4.0|
+---------+----------+------+------+-------------------+
only showing top 5 rows



In [13]:
# 1. Check if the 'prediction' (Cluster ID) column exists
df_clustered.select("lat", "lon", "prediction").show(5)

# 2. Count how many missions are in each of the 5 clusters
# This is great data to include in your report's "Results" section
df_clustered.groupBy("prediction").count().orderBy("prediction").show()

+---------+----------+----------+
|      lat|       lon|prediction|
+---------+----------+----------+
|  16.9025|106.014166|         3|
|19.602222|103.597222|         4|
|17.563611|105.756666|         3|
|12.280833|105.128888|         0|
|11.131666|105.498333|         0|
+---------+----------+----------+
only showing top 5 rows



[Stage 92:>                                                         (0 + 1) / 1]

+----------+------+
|prediction| count|
+----------+------+
|         0|536233|
|         1|155249|
|         2|424666|
|         3|671109|
|         4|117042|
+----------+------+



In [14]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Assemble Features
reg_assembler = VectorAssembler(
    inputCols=["planes", "MFUNC_DESC_Index", "Aircraft_Root_Index"], 
    outputCol="features_reg"
)
reg_final_data = reg_assembler.transform(df_clustered).select(
    F.col("features_reg"), 
    F.col("weight").alias("label")
)

# Train/Test Split
train, test = reg_final_data.randomSplit([0.7, 0.3], seed=42)

# Train Model
lr = LinearRegression(featuresCol="features_reg", labelCol="label", regParam=0.1)
lr_model = lr.fit(train)

# Results
rmse = RegressionEvaluator(metricName="rmse").evaluate(lr_model.transform(test))
print(f"Redo Complete. New RMSE: {rmse}")

26/03/10 13:32:26 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
[Stage 97:>                                                         (0 + 1) / 1]

Redo Complete. New RMSE: 371.04500311930354


In [15]:
# 1. Apply Clustering
k_model.setPredictionCol("Cluster_ID")
df_clustered = k_model.transform(df_final_input)

# 2. Apply Regression 
# Note: Regression output is 'prediction', Clustering is 'Cluster_ID'
df_results = lr_model.transform(df_clustered)

# 3. Select columns for Tableau using the confirmed root schema names
final_export = df_results.select(
    "Year","Month",
    "TgtCountry",
    "MilService",
    F.col("lat").alias("Latitude"),
    F.col("lon").alias("Longitude"),
    F.col("prediction").alias("Predicted_Weight"),
    F.col("weight").alias("Actual_Weight"),
    "Cluster_ID",
    "Aircraft_Root",
    F.col("planes").alias("Number_of_Aircraft"),
    "MFUNC_DESC"
)

# 4. Save
final_export.coalesce(1).write.mode("overwrite").option("header", "true") \
    .csv("file:///Users/anmolkhatri/Downloads/Vietnam_Final_Tableau_Data_1")

print("Export Complete. File is ready for Tableau.")

[Stage 98:>                                                         (0 + 1) / 1]

Export Complete. File is ready for Tableau.


In [17]:
from pyspark.ml.clustering import KMeans

# Assemble and Train
geo_assembler = VectorAssembler(inputCols=["lat", "lon"], outputCol="geo_features")
geo_data = geo_assembler.transform(df_ml_input)

kmeans = KMeans(featuresCol="geo_features", k=5, seed=42)
k_model = kmeans.fit(geo_data)

# Add Cluster IDs to the data
df_clustered = k_model.transform(geo_data)
print("Clusters assigned successfully.")
# 1. Check if the 'prediction' (Cluster ID) column exists
df_clustered.select("lat", "lon", "prediction").show(5)

# 2. Count how many missions are in each of the 5 clusters
# This is great data to include in your report's "Results" section
df_clustered.groupBy("prediction").count().orderBy("prediction").show()

ConnectionRefusedError: [Errno 61] Connection refused

In [16]:
df.printSchema()

root
 |-- SourceRecord: string (nullable = true)
 |-- SourceID: integer (nullable = true)
 |-- MSNDate: integer (nullable = true)
 |-- Conflict: string (nullable = true)
 |-- Theater: string (nullable = true)
 |-- CountryFlyingMission: string (nullable = true)
 |-- MilService: string (nullable = true)
 |-- NumberedAirForce: string (nullable = true)
 |-- AirForceGroup: string (nullable = true)
 |-- AirForceSQDN: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Callsign: string (nullable = true)
 |-- Aircraft_Original: string (nullable = true)
 |-- Aircraft_Root: string (nullable = true)
 |-- MissionID: string (nullable = true)
 |-- MFUNC: string (nullable = true)
 |-- MFUNC_DESC: string (nullable = true)
 |-- ServiceSupported: double (nullable = true)
 |-- OperationSupported: string (nullable = true)
 |-- TakeoffLocation: string (nullable = true)
 |-- TakeoffLatDeg: integer (nullable = true)
 |-- TakeoffLatDecimal: double (nullable = true)
 |-- TakeoffLatDD_DDD: double 